# 2. 사이킷런으로 시작하는 머신러닝
### CUAI 과제 (2.2 ~ 2.6)

> 사이킷런(scikit-learn)은 파이썬 머신러닝 라이브러리 중 가장 많이 사용되는 라이브러리.  
> 다양한 머신러닝 알고리즘과 데이터 전처리, 모델 평가 도구를 일관된 API로 제공한다.

---

## 2.2 첫 번째 머신러닝 만들어 보기 – 붓꽃 품종 예측하기

머신러닝의 기본 흐름을 익히기 위한 첫 예제.  
붓꽃(Iris) 데이터셋을 이용해 꽃받침/꽃잎의 크기로 품종을 분류하는 모델을 만들어본다.

**전체 흐름:**  
`데이터 로드` → `학습/테스트 분리` → `모델 학습(fit)` → `예측(predict)` → `평가(accuracy_score)`

In [ ]:
from sklearn.datasets import load_iris           # 붓꽃 데이터셋 로더
from sklearn.tree import DecisionTreeClassifier  # 결정 트리 분류기
from sklearn.model_selection import train_test_split  # 학습/테스트 분리 함수

In [ ]:
import pandas as pd

# 붓꽃 데이터셋 로드
# load_iris()는 Bunch 객체를 반환 - data, target, feature_names, target_names 등을 담고 있음
iris = load_iris()

iris_data = iris.data    # feature 데이터 (꽃받침 길이/너비, 꽃잎 길이/너비) - numpy array
iris_label = iris.target # 정답 레이블 (0: setosa, 1: versicolor, 2: virginica)

print('iris target값:', iris_label)
print('iris target명:', iris.target_names)

# 데이터를 보기 쉽게 DataFrame으로 변환
iris_df = pd.DataFrame(data=iris_data, columns=iris.feature_names)
iris_df['label'] = iris.target
iris_df.head(3)

In [ ]:
# 전체 데이터를 학습용 80% / 테스트용 20%로 분리
# random_state를 고정해야 실행할 때마다 같은 결과를 얻을 수 있음
X_train, X_test, y_train, y_test = train_test_split(iris_data, iris_label, 
                                                    test_size=0.2, random_state=11)

In [ ]:
# 결정 트리 분류기 객체 생성
# random_state 고정으로 트리 구성 시 동일한 결과 보장
dt_clf = DecisionTreeClassifier(random_state=11)

# fit() : 학습 데이터로 모델 학습
dt_clf.fit(X_train, y_train)

In [ ]:
# predict() : 학습된 모델로 테스트 데이터 예측
pred = dt_clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

# accuracy_score(실제값, 예측값) : 정확도 계산
# 정확도 = 맞게 예측한 수 / 전체 예측 수
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

---
## 2.3 사이킷런의 기반 프레임워크 익히기

사이킷런 데이터셋의 구조를 이해하는 섹션.  
`load_iris()`가 반환하는 Bunch 객체는 딕셔너리처럼 key로 접근할 수 있으며,  
`.data`, `.target`, `.feature_names`, `.target_names` 등의 속성을 갖는다.

In [ ]:
from sklearn.datasets import load_iris

iris_data = load_iris()
print(type(iris_data))  # sklearn.utils.Bunch 타입 → 딕셔너리와 유사한 구조

In [ ]:
# Bunch 객체가 어떤 키(속성)들을 가지고 있는지 확인
keys = iris_data.keys()
print('붓꽃 데이터 세트의 키들:', keys)

In [ ]:
# 각 속성의 타입과 형태 확인
# feature_names : 각 feature(컬럼)의 이름 리스트
print('\n feature_names 의 type:', type(iris_data.feature_names))
print(' feature_names 의 shape:', len(iris_data.feature_names))
print(iris_data.feature_names)

# target_names : 클래스 이름 (setosa, versicolor, virginica)
print('\n target_names 의 type:', type(iris_data.target_names))
print(' feature_names 의 shape:', len(iris_data.target_names))
print(iris_data.target_names)

# data : 실제 feature 값들 (150행 × 4열 numpy array)
print('\n data 의 type:', type(iris_data.data))
print(' data 의 shape:', iris_data.data.shape)
print(iris_data['data'])

# target : 각 샘플의 정답 레이블 (0, 1, 2 중 하나)
print('\n target 의 type:', type(iris_data.target))
print(' target 의 shape:', iris_data.target.shape)
print(iris_data.target)

---
## 2.4 Model Selection 모듈 소개

`sklearn.model_selection`은 모델의 학습/평가 방법을 제공하는 모듈.  
데이터를 어떻게 나누고, 어떻게 검증하느냐가 모델 성능 평가의 핵심이다.

---
### 학습 데이터로만 예측하면 안 되는 이유 (과적합 문제)

학습 데이터로 학습하고, 같은 학습 데이터로 예측하면 정확도가 비정상적으로 높게 나온다.  
이는 모델이 답을 '외운' 것이지, 실제로 잘 일반화된 것이 아니다 → 과적합(Overfitting)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
dt_clf = DecisionTreeClassifier()
train_data = iris.data
train_label = iris.target

# 학습 데이터로 그대로 학습
dt_clf.fit(train_data, train_label)

# 학습한 데이터로 다시 예측 → 정확도가 100%가 나오는 문제 발생
# 이건 모델이 답을 그냥 외워버린 것 (과적합)
pred = dt_clf.predict(train_data)
print('예측 정확도:', accuracy_score(train_label, pred))  # 1.0 나옴 → 신뢰 불가

### train_test_split() - 학습/테스트 데이터 분리

위 문제를 해결하기 위해 데이터를 학습용과 테스트용으로 나눠서 평가한다.  
테스트 데이터는 학습에 전혀 사용하지 않고, 오직 최종 성능 측정에만 사용.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

dt_clf = DecisionTreeClassifier()
iris_data = load_iris()

# 전체 데이터의 30%를 테스트용으로 분리
X_train, X_test, y_train, y_test = train_test_split(iris_data.data, iris_data.target, 
                                                    test_size=0.3, random_state=121)

In [ ]:
# 학습 데이터로만 학습하고, 테스트 데이터로 성능 평가
dt_clf.fit(X_train, y_train)
pred = dt_clf.predict(X_test)
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

### 교차 검증 (Cross Validation)

단순히 한 번만 train/test를 나누면 데이터 분할 방식에 따라 성능이 달라질 수 있음.  
교차 검증은 데이터를 여러 번 나눠서 평균 성능을 구해 더 신뢰할 수 있는 평가를 한다.

---
#### K-Fold 교차 검증

전체 데이터를 K개의 폴드(fold)로 나눈 뒤,  
K번 반복하면서 매번 다른 1개 폴드를 테스트셋으로, 나머지 K-1개를 학습셋으로 사용.

```
[fold1][fold2][fold3][fold4][fold5]
 테스트  학습   학습   학습   학습   → 1회
 학습   테스트  학습   학습   학습   → 2회
 ...                               → 총 K회
```

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
import numpy as np

iris = load_iris()
features = iris.data
label = iris.target
dt_clf = DecisionTreeClassifier(random_state=156)

# KFold 객체 생성 - 데이터를 5개 폴드로 분리
kfold = KFold(n_splits=5)
cv_accuracy = []  # 각 폴드별 정확도를 저장할 리스트
print('붓꽃 데이터 세트 크기:', features.shape[0])  # 150개 → 폴드당 30개씩

In [ ]:
n_iter = 0

# kfold.split(features) : 각 폴드의 학습/테스트 인덱스를 반환
for train_index, test_index in kfold.split(features):
    # 인덱스를 이용해 학습용/검증용 데이터 추출
    X_train, X_test = features[train_index], features[test_index]
    y_train, y_test = label[train_index], label[test_index]
    
    # 학습 및 예측
    dt_clf.fit(X_train, y_train)
    pred = dt_clf.predict(X_test)
    n_iter += 1
    
    # 각 폴드별 정확도 측정
    accuracy = np.round(accuracy_score(y_test, pred), 4)
    train_size = X_train.shape[0]
    test_size = X_test.shape[0]
    print('\n#{0} 교차 검증 정확도 :{1}, 학습 데이터 크기: {2}, 검증 데이터 크기: {3}'
          .format(n_iter, accuracy, train_size, test_size))
    print('#{0} 검증 세트 인덱스:{1}'.format(n_iter, test_index))
    cv_accuracy.append(accuracy)

# K번의 평균 정확도 → 더 신뢰할 수 있는 성능 지표
print('\n## 평균 검증 정확도:', np.mean(cv_accuracy))

#### Stratified K-Fold

일반 KFold는 클래스 비율을 고려하지 않고 분할하기 때문에,  
특정 폴드에 특정 클래스가 몰릴 수 있다는 문제가 있음.  

Stratified K-Fold는 각 폴드에서 클래스 비율을 원본과 동일하게 유지하며 분할.  
→ 불균형 데이터셋에서 특히 중요하다.

In [ ]:
import pandas as pd

iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['label'] = iris.target

# 클래스별 데이터 분포 확인 - 각 클래스(0,1,2)가 50개씩 균등 분포
iris_df['label'].value_counts()

In [ ]:
# 일반 KFold로 분할 시 클래스 분포가 불균형해지는 케이스 확인
kfold = KFold(n_splits=3)
n_iter = 0
for train_index, test_index in kfold.split(iris_df):
    n_iter += 1
    label_train = iris_df['label'].iloc[train_index]
    label_test = iris_df['label'].iloc[test_index]
    print('## 교차 검증: {0}'.format(n_iter))
    print('학습 레이블 데이터 분포:\n', label_train.value_counts())
    print('검증 레이블 데이터 분포:\n', label_test.value_counts())
    # → 검증 세트에 특정 클래스만 몰리는 현상 발생 가능

In [ ]:
from sklearn.model_selection import StratifiedKFold

# StratifiedKFold : split() 호출 시 반드시 레이블 데이터도 함께 전달해야 함
skf = StratifiedKFold(n_splits=3)
n_iter = 0

for train_index, test_index in skf.split(iris_df, iris_df['label']):
    n_iter += 1
    label_train = iris_df['label'].iloc[train_index]
    label_test = iris_df['label'].iloc[test_index]
    print('## 교차 검증: {0}'.format(n_iter))
    print('학습 레이블 데이터 분포:\n', label_train.value_counts())
    print('검증 레이블 데이터 분포:\n', label_test.value_counts())
    # → 모든 폴드에서 클래스 비율이 동일하게 유지됨

In [ ]:
dt_clf = DecisionTreeClassifier(random_state=156)
skfold = StratifiedKFold(n_splits=3)
n_iter = 0
cv_accuracy = []

# Stratified K-Fold로 실제 학습 및 평가 수행
for train_index, test_index in skfold.split(features, label):
    X_train, X_test = features[train_index], features[test_index]
    y_train, y_test = label[train_index], label[test_index]
    
    dt_clf.fit(X_train, y_train)
    pred = dt_clf.predict(X_test)

    n_iter += 1
    accuracy = np.round(accuracy_score(y_test, pred), 4)
    train_size = X_train.shape[0]
    test_size = X_test.shape[0]
    print('\n#{0} 교차 검증 정확도 :{1}, 학습 데이터 크기: {2}, 검증 데이터 크기: {3}'
          .format(n_iter, accuracy, train_size, test_size))
    print('#{0} 검증 세트 인덱스:{1}'.format(n_iter, test_index))
    cv_accuracy.append(accuracy)

print('\n## 교차 검증별 정확도:', np.round(cv_accuracy, 4))
print('## 평균 검증 정확도:', np.round(np.mean(cv_accuracy), 4))

#### cross_val_score() - 교차 검증을 더 간편하게

위의 KFold 반복 과정을 한 줄로 처리해주는 편의 함수.  
내부적으로 StratifiedKFold를 사용하므로 분류 문제에서는 클래스 비율도 자동으로 유지된다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.datasets import load_iris

iris_data = load_iris()
dt_clf = DecisionTreeClassifier(random_state=156)

data = iris_data.data
label = iris_data.target

# cross_val_score(모델, feature, label, scoring, cv)
# 내부적으로 분할 → 학습 → 예측 → 평가 를 cv번 수행 후 각 점수 반환
scores = cross_val_score(dt_clf, data, label, scoring='accuracy', cv=3)
print('교차 검증별 정확도:', np.round(scores, 4))
print('평균 검증 정확도:', np.round(np.mean(scores), 4))

#### GridSearchCV - 하이퍼파라미터 자동 탐색

모델의 하이퍼파라미터(max_depth, min_samples_split 등)를 직접 조정하면 성능이 달라진다.  
어떤 값이 최적인지 모를 때, 가능한 모든 조합을 교차 검증으로 평가하는 것이 GridSearchCV.

- `param_grid` : 탐색할 파라미터 조합
- `cv` : 각 조합에 적용할 교차 검증 횟수
- `refit=True` : 최적 파라미터로 전체 데이터에 재학습 (기본값)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris_data.data, iris_data.target, 
                                                    test_size=0.2, random_state=121)
dtree = DecisionTreeClassifier()

# 탐색할 하이퍼파라미터 범위를 딕셔너리로 정의
# max_depth 3가지 × min_samples_split 2가지 = 총 6가지 조합 × cv=3 → 18번 학습
parameters = {'max_depth': [1, 2, 3], 'min_samples_split': [2, 3]}

In [ ]:
import pandas as pd

# GridSearchCV 객체 생성 후 학습
# refit=True(기본값) → 최적 파라미터로 자동 재학습
grid_dtree = GridSearchCV(dtree, param_grid=parameters, cv=3, refit=True)
grid_dtree.fit(X_train, y_train)

# 교차 검증 결과를 DataFrame으로 정리해서 확인
scores_df = pd.DataFrame(grid_dtree.cv_results_)
scores_df[['params', 'mean_test_score', 'rank_test_score',
           'split0_test_score', 'split1_test_score', 'split2_test_score']]

In [ ]:
# 최적 파라미터와 해당 교차 검증 평균 정확도 확인
print('GridSearchCV 최적 파라미터:', grid_dtree.best_params_)
print('GridSearchCV 최고 정확도: {0:.4f}'.format(grid_dtree.best_score_))

In [ ]:
# best_estimator_ : 최적 파라미터로 refit된 모델 → 바로 예측에 사용 가능
estimator = grid_dtree.best_estimator_

pred = estimator.predict(X_test)
print('테스트 데이터 세트 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

---
## 2.5 데이터 전처리 (Data Preprocessing)

머신러닝 모델은 숫자만 입력받기 때문에, 문자형 데이터는 반드시 숫자로 변환해야 한다.  
또한 피처들의 스케일이 크게 다르면 특정 피처에 편향된 학습이 이루어질 수 있어 정규화가 필요하다.

---
### 데이터 인코딩
#### 레이블 인코딩 (Label Encoding)

문자열 카테고리를 0, 1, 2, ... 순서의 정수로 변환.  
단점: 숫자의 크기가 순서/크기 관계를 암시할 수 있어 트리 계열 외 모델에서 문제 발생 가능.

In [ ]:
from sklearn.preprocessing import LabelEncoder

items = ['TV', '냉장고', '전자레인지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

# LabelEncoder 생성 → fit(학습) → transform(변환)
# fit 단계에서 알파벳/가나다 순으로 정렬해서 0부터 번호 부여
encoder = LabelEncoder()
encoder.fit(items)
labels = encoder.transform(items)
print('인코딩 변환값:', labels)

In [ ]:
# classes_ : fit 시 학습한 클래스 목록 (정렬된 순서로 인덱스 부여됨)
print('인코딩 클래스:', encoder.classes_)

In [ ]:
# inverse_transform : 인코딩된 숫자를 다시 원본 문자열로 복원
print('디코딩 원본 값:', encoder.inverse_transform([4, 5, 2, 0, 1, 1, 3, 3]))

#### 원-핫 인코딩 (One-Hot Encoding)

레이블 인코딩의 숫자 크기 문제를 해결하기 위한 방법.  
각 카테고리를 별도의 이진 컬럼으로 분리하여 해당 카테고리면 1, 아니면 0으로 표현.  
→ 카테고리 간 순서/크기 관계가 생기지 않음.

```
TV     → [1, 0, 0, 0, 0, 0]
냉장고  → [0, 1, 0, 0, 0, 0]
...
```

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

items = ['TV', '냉장고', '전자레인지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

# OneHotEncoder는 2차원 배열을 입력받음 → reshape(-1, 1)로 변환
items = np.array(items).reshape(-1, 1)

oh_encoder = OneHotEncoder()
oh_encoder.fit(items)
oh_labels = oh_encoder.transform(items)

# 결과는 희소 행렬(sparse matrix)로 반환됨 → toarray()로 밀집 행렬로 변환
print('원-핫 인코딩 데이터')
print(oh_labels.toarray())
print('원-핫 인코딩 데이터 차원')
print(oh_labels.shape)  # (8행, 6열) → 카테고리 수만큼 컬럼 생성

In [ ]:
import pandas as pd

# pandas의 get_dummies()를 쓰면 훨씬 간편하게 원-핫 인코딩 가능
# 문자열 컬럼을 자동으로 감지해서 인코딩
df = pd.DataFrame({'item': ['TV', '냉장고', '전자레인지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']})
pd.get_dummies(df)

### 피처 스케일링과 정규화

피처마다 값의 범위(스케일)가 다르면, 스케일이 큰 피처가 모델에 과도한 영향을 줄 수 있다.  
→ 모든 피처를 비슷한 범위로 맞춰주는 스케일링이 필요.

---
#### StandardScaler

각 피처의 값을 평균=0, 표준편차=1이 되도록 변환.  
정규분포를 따른다고 가정하는 알고리즘(SVM, 선형회귀 등)에 적합.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
iris_data = iris.data
iris_df = pd.DataFrame(data=iris_data, columns=iris.feature_names)

# 스케일링 전 평균/분산 확인 → 피처마다 값 범위가 다름
print('feature 들의 평균 값')
print(iris_df.mean())
print('\nfeature 들의 분산 값')
print(iris_df.var())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(iris_df)              # 평균과 표준편차 계산
iris_scaled = scaler.transform(iris_df)  # 계산된 값으로 변환

iris_df_scaled = pd.DataFrame(data=iris_scaled, columns=iris.feature_names)

# 스케일링 후 평균 ≈ 0, 분산 ≈ 1 이 되었는지 확인
print('feature 들의 평균 값')
print(iris_df_scaled.mean())
print('\nfeature 들의 분산 값')
print(iris_df_scaled.var())

#### MinMaxScaler

각 피처의 값을 최솟값=0, 최댓값=1 범위로 변환.  
이상치(outlier)에 민감하지만, 값의 범위를 명확히 0~1로 제한해야 할 때 사용.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(iris_df)
iris_scaled = scaler.transform(iris_df)

iris_df_scaled = pd.DataFrame(data=iris_scaled, columns=iris.feature_names)

# 모든 피처의 최솟값 = 0, 최댓값 = 1 인지 확인
print('feature들의 최솟값')
print(iris_df_scaled.min())
print('\nfeature들의 최댓값')
print(iris_df_scaled.max())

#### Scaler 사용 시 주의사항 - fit은 학습 데이터에만!

 흔한 실수: 테스트 데이터에 fit()을 다시 호출하는 것.  
→ 테스트 데이터의 스케일 기준이 달라져서 학습/테스트 간 데이터 불일치 발생.

올바른 사용법:
1. 학습 데이터로 `fit()` → 스케일 기준 학습
2. 학습 데이터에 `transform()` 적용
3. 테스트 데이터는 fit() 없이 `transform()`만 적용 (학습 데이터 기준으로 변환)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 학습 데이터: 0~10 / 테스트 데이터: 0~5
train_array = np.arange(0, 11).reshape(-1, 1)
test_array  = np.arange(0, 6).reshape(-1, 1)

In [ ]:
scaler = MinMaxScaler()

# 학습 데이터 기준으로 fit (최솟값=0, 최댓값=10 학습)
scaler.fit(train_array)

# 학습 데이터 변환: 0→0, 10→1 (1/10 스케일)
train_scaled = scaler.transform(train_array)
print('원본 train_array 데이터:', np.round(train_array.reshape(-1), 2))
print('Scale된 train_array 데이터:', np.round(train_scaled.reshape(-1), 2))

In [ ]:
# 잘못된 방법: 테스트 데이터에 fit()을 다시 호출
# → 최솟값=0, 최댓값=5 기준으로 재설정됨 → 학습 데이터와 다른 기준 사용
scaler.fit(test_array)
test_scaled = scaler.transform(test_array)

print('원본 test_array 데이터:', np.round(test_array.reshape(-1), 2))
print('Scale된 test_array 데이터:', np.round(test_scaled.reshape(-1), 2))
# 5가 1.0이 되어버림 → 학습 데이터에서 10이 1.0이었는데 불일치!

In [ ]:
# 올바른 방법: 학습 데이터로 fit 후, 테스트 데이터는 transform만
scaler = MinMaxScaler()
scaler.fit(train_array)                      # 학습 데이터 기준으로만 fit
train_scaled = scaler.transform(train_array)
print('원본 train_array 데이터:', np.round(train_array.reshape(-1), 2))
print('Scale된 train_array 데이터:', np.round(train_scaled.reshape(-1), 2))

# 테스트 데이터는 fit 없이 transform만 → 학습 데이터와 동일한 기준 적용
# 5는 10 기준 스케일에서 0.5가 됨 (일관성 유지)
test_scaled = scaler.transform(test_array)
print('\n원본 test_array 데이터:', np.round(test_array.reshape(-1), 2))
print('Scale된 test_array 데이터:', np.round(test_scaled.reshape(-1), 2))

---
## 2.6 사이킷런으로 수행하는 타이타닉 생존자 예측

지금까지 배운 내용(전처리, 모델 학습, 교차 검증, GridSearchCV)을 실제 데이터에 종합 적용.  
타이타닉 생존자 예측은 머신러닝 입문용 클래식 과제로,  
승객의 나이, 성별, 객실 등급 등의 정보로 생존 여부를 분류한다.

주요 컬럼:
- `Survived` : 생존 여부 (0=사망, 1=생존) → 정답 레이블
- `Pclass` : 객실 등급 (1=1등급, 2=2등급, 3=3등급)
- `Sex` : 성별
- `Age` : 나이 (결측치 존재)
- `Cabin` : 객실 번호 (결측치 매우 많음)
- `Embarked` : 탑승 항구 (C/Q/S)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 타이타닉 학습 데이터 로드
titanic_df = pd.read_csv('./titanic_train.csv')
titanic_df.head(3)

In [ ]:
# 데이터 기본 정보 확인
# → 컬럼별 타입, Non-Null Count를 보고 결측치 있는 컬럼 파악
print('\n ### train 데이터 정보 ###  \n')
print(titanic_df.info())

### Step 1. 결측치(Null) 처리

머신러닝 모델은 NaN을 처리할 수 없으므로 반드시 결측치를 채워줘야 한다.
- `Age` : 숫자형 → 평균값으로 대체
- `Cabin`, `Embarked` : 문자형 → 'N'으로 대체

In [ ]:
# fillna() : 결측치를 지정한 값으로 채움
# inplace=True : 원본 DataFrame을 직접 수정
titanic_df['Age'].fillna(titanic_df['Age'].mean(), inplace=True)  # 평균 나이로 대체
titanic_df['Cabin'].fillna('N', inplace=True)     # 객실 정보 없음 → 'N'
titanic_df['Embarked'].fillna('N', inplace=True)  # 탑승 항구 없음 → 'N'

# 결측치가 모두 처리됐는지 확인
print('데이터 세트 Null 값 갯수 ', titanic_df.isnull().sum().sum())

In [ ]:
# 주요 카테고리 컬럼들의 값 분포 확인
# → 인코딩 전에 어떤 값들이 있는지 파악하는 EDA 과정
print(' Sex 값 분포 :\n', titanic_df['Sex'].value_counts())
print('\n Cabin 값 분포 :\n', titanic_df['Cabin'].value_counts())
print('\n Embarked 값 분포 :\n', titanic_df['Embarked'].value_counts())

In [ ]:
# Cabin은 'C85', 'B28' 처럼 알파벳+숫자 형태
# 앞 알파벳(구역)만 추출해서 사용 (세부 호수보다 구역이 더 의미있는 feature)
titanic_df['Cabin'] = titanic_df['Cabin'].str[:1]
print(titanic_df['Cabin'].head(3))

### Step 2. 탐색적 데이터 분석 (EDA)

어떤 피처가 생존과 연관이 있는지 시각화로 파악.  
이를 통해 피처 엔지니어링 방향을 결정할 수 있다.

In [ ]:
# 성별과 생존의 관계
titanic_df.groupby(['Sex', 'Survived'])['Survived'].count()

In [ ]:
# 성별별 생존율 시각화
# → 여성의 생존율이 남성보다 훨씬 높음 (여성/어린이 우선 대피)
sns.barplot(x='Sex', y='Survived', data=titanic_df)

In [ ]:
# 객실 등급(Pclass)과 성별을 동시에 고려한 생존율
# → 1등급 여성의 생존율이 가장 높고, 3등급 남성이 가장 낮음
sns.barplot(x='Pclass', y='Survived', hue='Sex', data=titanic_df)

In [ ]:
# 나이 그룹별 생존율 시각화
# 나이를 그룹으로 묶어서 분석하기 위한 카테고리화 함수
def get_category(age):
    cat = ''
    if age <= -1:   cat = 'Unknown'
    elif age <= 5:  cat = 'Baby'
    elif age <= 12: cat = 'Child'
    elif age <= 18: cat = 'Teenager'
    elif age <= 25: cat = 'Student'
    elif age <= 35: cat = 'Young Adult'
    elif age <= 60: cat = 'Adult'
    else:           cat = 'Elderly'
    return cat

plt.figure(figsize=(10, 6))
group_names = ['Unknown', 'Baby', 'Child', 'Teenager', 'Student', 'Young Adult', 'Adult', 'Elderly']

# apply(lambda) : 각 행의 Age 값을 get_category() 함수에 넣어 카테고리 반환
titanic_df['Age_cat'] = titanic_df['Age'].apply(lambda x: get_category(x))
sns.barplot(x='Age_cat', y='Survived', hue='Sex', data=titanic_df, order=group_names)

# 시각화용으로만 사용한 임시 컬럼 삭제
titanic_df.drop('Age_cat', axis=1, inplace=True)

### Step 3. 데이터 전처리 파이프라인 구성

재사용 가능하도록 전처리 과정을 함수로 모듈화.  
학습 데이터와 테스트 데이터에 동일한 전처리를 적용하기 위함이다.

In [ ]:
from sklearn import preprocessing

# 레이블 인코딩 함수 - Cabin, Sex, Embarked 컬럼을 숫자로 변환
def encode_features(dataDF):
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = preprocessing.LabelEncoder()
        le = le.fit(dataDF[feature])
        dataDF[feature] = le.transform(dataDF[feature])
    return dataDF

titanic_df = encode_features(titanic_df)
titanic_df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ① 결측치 처리 함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# ② 불필요한 피처 제거 함수
# PassengerId, Name, Ticket은 생존 예측에 직접적 도움이 되지 않는 식별자 컬럼
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# ③ 레이블 인코딩 함수
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]  # Cabin 구역만 추출
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# ④ 전체 전처리를 순서대로 호출하는 통합 함수
def transform_features(df):
    df = fillna(df)           # 1. 결측치 처리
    df = drop_features(df)    # 2. 불필요 컬럼 제거
    df = format_features(df)  # 3. 인코딩
    return df

In [ ]:
# 원본 데이터를 다시 로드하고 feature/label 분리
# (위에서 이미 전처리가 된 데이터를 재사용하지 않기 위해 다시 로드)
titanic_df = pd.read_csv('./titanic_train.csv')

y_titanic_df = titanic_df['Survived']                # 정답 레이블
X_titanic_df = titanic_df.drop('Survived', axis=1)   # feature 데이터

# 전처리 적용
X_titanic_df = transform_features(X_titanic_df)

In [ ]:
from sklearn.model_selection import train_test_split

# 학습/테스트 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df,
                                                    test_size=0.2, random_state=11)

### Step 4. 여러 모델로 학습 및 성능 비교

같은 데이터에 다양한 알고리즘을 적용해보고 성능을 비교.
- DecisionTree : 단순하지만 해석 가능
- RandomForest : 여러 트리의 앙상블 → 일반적으로 더 성능 좋음
- LogisticRegression : 선형 기반의 이진 분류 모델

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 세 가지 분류기 생성
dt_clf = DecisionTreeClassifier(random_state=11)
rf_clf = RandomForestClassifier(random_state=11)
lr_clf = LogisticRegression(solver='liblinear')  # 소규모 데이터셋에 적합한 solver

# DecisionTreeClassifier 학습/예측/평가
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)
print('DecisionTreeClassifier 정확도: {0:.4f}'.format(accuracy_score(y_test, dt_pred)))

# RandomForestClassifier 학습/예측/평가
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
print('RandomForestClassifier 정확도:{0:.4f}'.format(accuracy_score(y_test, rf_pred)))

# LogisticRegression 학습/예측/평가
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
print('LogisticRegression 정확도: {0:.4f}'.format(accuracy_score(y_test, lr_pred)))

### Step 5. KFold 교차 검증으로 성능 재확인

단일 train/test 분할은 분할 방식에 따라 결과가 달라질 수 있어,  
KFold 교차 검증으로 더 안정적인 성능 수치를 확인한다.

In [ ]:
from sklearn.model_selection import KFold

def exec_kfold(clf, folds=5):
    kfold = KFold(n_splits=folds)
    scores = []
    
    for iter_count, (train_index, test_index) in enumerate(kfold.split(X_titanic_df)):
        # 교차 검증 인덱스로 데이터 분할
        X_train, X_test = X_titanic_df.values[train_index], X_titanic_df.values[test_index]
        y_train, y_test = y_titanic_df.values[train_index], y_titanic_df.values[test_index]
        
        clf.fit(X_train, y_train)
        predictions = clf.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        scores.append(accuracy)
        print("교차 검증 {0} 정확도: {1:.4f}".format(iter_count, accuracy))
    
    mean_score = np.mean(scores)
    print("평균 정확도: {0:.4f}".format(mean_score))

# DecisionTree에 KFold 교차 검증 적용
exec_kfold(dt_clf, folds=5)

### Step 6. GridSearchCV로 최적 하이퍼파라미터 탐색

DecisionTree의 하이퍼파라미터를 Grid Search로 최적화하여 성능을 더 끌어올린다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 탐색할 파라미터 범위 설정
# max_depth 4개 × min_samples_split 3개 × min_samples_leaf 3개 = 36가지 조합
parameters = {'max_depth': [2, 3, 5, 10],
              'min_samples_split': [2, 3, 5],
              'min_samples_leaf': [1, 5, 8]}

# 5-Fold 교차 검증으로 각 조합 평가
grid_dclf = GridSearchCV(dt_clf, param_grid=parameters, scoring='accuracy', cv=5)
grid_dclf.fit(X_train, y_train)

print('GridSearchCV 최적 하이퍼 파라미터 :', grid_dclf.best_params_)
print('GridSearchCV 최고 정확도: {0:.4f}'.format(grid_dclf.best_score_))

# 최적 파라미터로 학습된 모델로 테스트 데이터 예측
best_dclf = grid_dclf.best_estimator_
dpredictions = best_dclf.predict(X_test)
accuracy = accuracy_score(y_test, dpredictions)
print('테스트 세트에서의 DecisionTreeClassifier 정확도 : {0:.4f}'.format(accuracy))